# Remove the Glider Motion from the DVL Velocities
Without bottom tracking, the glider DVL measures the water velocity **relative to the glider**. To get the absolute water velocity, the glider's motion over the ground has to be removed. For each dive, the water velocity profile and the glider's velocity over the ground are solved for together by least squares (as for a lowered ADCP; Visbeck, 2002; Todd et al., 2017):

- the **DVL** cells give the vertical shear,
- the **flight model** ([gliderflight](https://github.com/smerckel/gliderflight); Merckelbach et al., 2019) links the glider's velocity over the ground to the current at its depth,
- a **reference** sets the absolute level, which the DVL and the flight model can't: either the **depth-averaged current** from the GPS and dead reckoning (notebook 2), or **bottom tracking**.

The profiles are constructed both ways, independently, and each is checked against the other:

1. **Method 1**, referenced to the depth-averaged current, is checked against bottom tracking
2. **Method 2**, referenced to bottom tracking only, is checked against the depth-averaged current, and against held-out bottom tracking

Along the way, the flight model is checked against the depth rate on the dives and the climbs, against the DVL, and against the glider's speed through the water from bottom tracking; the frame of the velocities against the GPS; and the glider's own speed estimate (which the depth-averaged current depends on) against the flight model.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from ooicgsn_glider_dvl import bottom_track, flight, inversion
from ooicgsn_glider_dvl.binning import bin_profiles
from ooicgsn_glider_dvl.corrections import correct_vertical_velocity, shift_time
from ooicgsn_glider_dvl.merge import adcp_attitude_to_degrees, true_north_velocities

%matplotlib inline

In [ ]:
# CP05MOAS-GL379 deployment 8, deployed at the Pioneer NES array from 2019-09-27 to 2019-11-19
refdes = 'CP05MOAS-GL379-01-ADCPAM000'
eng_refdes = 'CP05MOAS-GL379-00-ENG000000'
deployment = 8
glider_name = '-'.join(refdes.split('-')[:2])
dep_str = f"{deployment:04d}"

# The glider's mass (kg), weighed before the deployment
glider_mass = 65.480

# Colors of the sources compared in the figures
DAC_COLOR, BT_COLOR, OBS_COLOR = '#2a78d6', '#eb6834', '#1baf7a'
BT_VARIABLES = ('bottom_track_eastward_velocity', 'bottom_track_northward_velocity')

### Load the Data
The merged (ADCP) dataset, the glider dataset (through the dives, climbs, and surfacings), and the depth-averaged currents from notebook 2, the bottom-track data, and the glider engineering variables the flight model needs (pitch, heading, buoyancy change, and the glider's own speed estimate). In this notebook, `glider` is the merged ADCP dataset, on the DVL times (the dives only), and `glider_ts` is the glider dataset, on the engineering times (the dives and the climbs).

Two corrections to the OOINet ADCP data are applied here for datasets merged before notebook 2 made them (both are skipped if they were already made):
- The ADCP heading, pitch, and roll are converted from hundredths of a degree to degrees.
- The water velocities are built from the raw ADCP velocities (`water_velocity_east/north`), not OOINet's `eastward/northward_seawater_velocity`, which OOINet rotated by the magnetic declination. The glider ADCP has no compass of its own and uses the glider's heading, which is already relative to true north (the glider dead reckons with it directly in its true north coordinates), so the raw velocities are already relative to true north and OOINet's rotation is a double correction. OOINet's velocities are kept as `ooi_eastward/northward_seawater_velocity` for reference. The bottom-track velocities and the glider's heading were never rotated, so all three are in the same, true north, frame. This is checked against the GPS below.

In [ ]:
glider = xr.open_dataset(f"../data/processed/{glider_name}.deployment{dep_str}.merged.nc").load()
glider = adcp_attitude_to_degrees(glider)
glider = true_north_velocities(glider)
glider_ts = xr.open_dataset(f"../data/processed/{glider_name}.deployment{dep_str}.glider.nc").load()

surfacings = pd.read_csv(
    f"../data/processed/{glider_name}.deployment{dep_str}.depth_averaged_current.csv",
    parse_dates=['dive_start_time', 'surface_time'],
)
bt = xr.open_dataset(f"../data/raw/{refdes}.deployment{dep_str}.adcp_bottom_track.nc")
# Correct the bottom track times for the DVL time offset, as the ADCP times were in notebook 2
bt = shift_time(bt, glider.attrs['dvl_time_offset'])
eng = pd.read_csv(
    f"../data/raw/deployment{dep_str}_{eng_refdes}-recovered_host-glider_eng_recovered.csv",
    usecols=flight.FLIGHT_VARIABLES,
)

### Calibrate the Flight Model
The model is calibrated against the depth rate during steady flight (deeper than 10 m, more than 5 m from the deepest depth of the dive or climb, pitched more than 15 degrees, and not pumping), for the glider's volume (`Vg`) and drag coefficient (`Cd0`), for the whole deployment and for each week.

It is calibrated on the glider dataset, with the **dives and the climbs**. On the dives alone, the volume and the drag trade off against each other: many pairs fit the dives' depth rate equally well, but they differ on the climbs, where the buoyancy is the other way. The climbs pin them down.

The depth rate only constrains the glider's vertical velocity. Its horizontal velocity, which the profiles depend on, follows from its glide angle: its pitch plus the angle of attack. With the volume and drag fitted on the dives and the climbs, the model's horizontal speed on the dives is too fast, compared with the DVL and with bottom tracking (check 2), and no lift coefficient fixes it. An offset of the glider's reported pitch does: it changes the glide angle the other way on the dives and the climbs. `calibrate_pitch_offset` finds the offset at which the model's horizontal speed on the dives matches the DVL's (from its second cell; the first reads slightly low), calibrating the volume and drag on the dives and climbs for each trial offset. The lift coefficient is left at the gliderflight default.

The calibrated model is then solved on both datasets: on the DVL times for the profiles, and on the glider dataset for the depth-averaged current check.

In [ ]:
glider = flight.add_flight_variables(glider, eng)
steady = flight.calibration_mask(glider)
steady_ts = flight.calibration_mask(glider_ts, depth='m_depth', profile='science_profile_id')
dives = (glider_ts['phase'] == 1).values
climbs = (glider_ts['phase'] == 2).values
print(f"Steady flight: {steady.mean():.0%} of the DVL measurements; in the glider dataset, "
      f"{steady_ts[dives].mean():.0%} of the dives and {steady_ts[climbs].mean():.0%} of the climbs")

pitch_offset, pitch_trials = flight.calibrate_pitch_offset(
    glider, glider_ts, mass=glider_mass, steady=steady, glider_steady=steady_ts
)
print(f"Calibrated pitch offset: {pitch_offset:.2f} degrees")
pitch_trials[['pitch_offset', 'Vg', 'Cd0', 'rms_dhdt', 'forward_bias']].round(4)

In [ ]:
calibration = flight.calibrate_flight_model(
    glider_ts, mass=glider_mass, steady=steady_ts, window='7D', pitch_offset=pitch_offset
)
glider = flight.solve_flight_model(glider, calibration)
glider_ts = flight.solve_flight_model(glider_ts, calibration)
calibration

#### The Climbs Matter
The model's misfit to the depth rate (the depth rate minus the modeled vertical velocity) should average to zero on both the dives and the climbs. The calibration is compared with the same calibration without the pitch offset, and with one made with only the dives (the DVL times), with the model's mean horizontal speed on the dives minus the DVL's.

In [ ]:
dvl_speed = flight.dvl_glider_velocity(glider.copy(), cell=1)['dvl_glider_forward_speed'].values
versions = {
    'dives and climbs, pitch offset': calibration,
    'dives and climbs, no pitch offset': flight.calibrate_flight_model(glider_ts, mass=glider_mass, steady=steady_ts),
    'dives only, no pitch offset': flight.calibrate_flight_model(glider, mass=glider_mass, steady=steady),
}
rows = []
for name, version in versions.items():
    residuals = flight.depth_rate_residuals(flight.solve_flight_model(glider_ts.copy(), version), steady_ts)
    model = flight.solve_flight_model(glider.copy(), version)
    heading = np.deg2rad(model['vehicle_heading'].values)
    forward = (model['glider_eastward_velocity_through_water'].values * np.sin(heading)
               + model['glider_northward_velocity_through_water'].values * np.cos(heading))
    valid = steady & np.isfinite(forward) & np.isfinite(dvl_speed)
    rows.append({
        'calibration': name,
        'pitch_offset': version['pitch_offset'].iloc[0],
        'Vg': version['Vg'].iloc[0],
        'Cd0': version['Cd0'].iloc[0],
        'dive bias (cm/s)': residuals.loc['dive', 'bias'] * 100,
        'climb bias (cm/s)': residuals.loc['climb', 'bias'] * 100,
        'dive rms (cm/s)': residuals.loc['dive', 'rms'] * 100,
        'climb rms (cm/s)': residuals.loc['climb', 'rms'] * 100,
        'horizontal speed minus DVL (cm/s)': (forward[valid] - dvl_speed[valid]).mean() * 100,
    })
pd.DataFrame(rows).set_index('calibration').round(3)

### Check 1: the Flight Model Against the DVL
Close to the glider, the water moves with nearly the same current as at the glider, so the opposite of the DVL relative velocity is the glider's velocity through the water. The second cell (about 5 m below the glider) is used: the first reads slightly low (check 2). The pitch offset was calibrated so that the mean horizontal speed matches this cell's, so the mean forward speed agrees by construction; the eastward, northward, and upward velocities and the correlations are independent. The glider's own speed estimate (`m_speed`), which its dead reckoning (and so the depth-averaged current) uses, is compared too.

In [ ]:
glider = flight.dvl_glider_velocity(glider, cell=1)
display(flight.flight_model_skill(glider, steady).round(3))

model_speed = np.hypot(glider['glider_eastward_velocity_through_water'], glider['glider_northward_velocity_through_water'])
valid = steady & np.isfinite(model_speed.values) & np.isfinite(glider['dvl_glider_forward_speed'].values)
print(f"Mean horizontal speed through the water during steady flight:")
print(f"  flight model {float(model_speed[valid].mean()):.3f} m/s")
print(f"  DVL          {float(glider['dvl_glider_forward_speed'][valid].mean()):.3f} m/s")
print(f"  m_speed      {float(glider['dead_reckoning_speed'][valid].mean()):.3f} m/s")

### Bottom Tracking
The bottom-track velocities (converted to m/s) are subtracted from the DVL water velocities and binned on the same 2 m depth bins as the profiles, to check them with.

The DVL only tracks the bottom when it is within range, near the end of each dive on the shelf, so bottom tracking covers only the deepest part of each dive. Everything checked against it (checks 2, 5, and 6) describes the lower part of the water column.

In [ ]:
glider = bottom_track.add_bottom_track(glider, bt)
# Correct the bottom track upward velocity for the sound speed, as the water velocities were
glider = correct_vertical_velocity(
    glider,
    velocities=(
        'bottom_track_eastward_velocity',
        'bottom_track_northward_velocity',
        'bottom_track_upward_velocity',
    ),
    raw=None,
    mount_angle=11,
)
glider = bottom_track.remove_bottom_track(glider)

bt_profiles = bin_profiles(
    glider, velocity_variables=bottom_track.BOTTOM_TRACK_VELOCITY_VARIABLES, science_variables=()
)

In [ ]:
has_bt = np.isfinite(glider['bottom_track_eastward_velocity'].values)
coverage = pd.DataFrame({'profile_id': glider['profile_id'].values,
                         'depth': glider['depth_from_pressure'].values, 'bt': has_bt})
by_dive = coverage.groupby('profile_id')
dives_bt = pd.DataFrame({
    'max_depth': by_dive['depth'].max(),
    'fraction': by_dive['bt'].mean(),
    'top': coverage[coverage['bt']].groupby('profile_id')['depth'].min(),
})
with_bt = dives_bt.dropna()
print(f"{len(with_bt)} of {len(dives_bt)} dives have bottom tracking; "
      f"{has_bt.mean():.0%} of all ensembles")
print(f"In those dives, bottom tracking covers the deepest "
      f"{(with_bt['max_depth'] - with_bt['top']).median():.0f} m (median), starting at "
      f"{(with_bt['top'] / with_bt['max_depth']).median():.0%} of the dive's depth")

### Check 2: the Speed Through the Water from Bottom Tracking
Where the bottom is in range, the glider's velocity through the water can be measured without the flight model or the DVL cell nearest the glider: it is the glider's velocity over the ground (bottom tracking) minus the absolute water velocity at its depth, which the cells of earlier ensembles measured before the glider descended into that water (Merckelbach et al., 2019). The profiles are solved from the DVL and bottom tracking alone (`dac_weight=0`, `flight_weight=0`), and the speed along the glider's heading is compared with the flight model, the glider's own estimate (`m_speed`), and the DVL's first few cells (the opposite of their relative velocity). The pitch offset was calibrated with the DVL's second cell, not with bottom tracking, so this checks it independently.

In [ ]:
_, ground_bt_only = inversion.invert_profiles(
    glider, steady=steady, dac_weight=0, flight_weight=0, bottom_track_variables=BT_VARIABLES
)
through_water = bottom_track.through_water_velocity(glider, ground_bt_only)
through_water = through_water[np.isin(through_water['time'].values, glider['time'].values[steady])]
index = np.searchsorted(glider['time'].values, through_water['time'].values)

heading = np.deg2rad(glider['vehicle_heading'].values[index])
estimates = {
    'flight model': (glider['glider_eastward_velocity_through_water'].values[index] * np.sin(heading)
                     + glider['glider_northward_velocity_through_water'].values[index] * np.cos(heading)),
    'm_speed': glider['dead_reckoning_speed'].values[index],
}
for cell in range(4):
    dvl = flight.dvl_glider_velocity(glider.copy(), cell=cell)
    estimates[f'DVL cell {cell + 1}'] = dvl['dvl_glider_forward_speed'].values[index]

rows = []
for name, values in estimates.items():
    valid = np.isfinite(values)
    difference = values[valid] - through_water['forward'].values[valid]
    rows.append({'estimate': name, 'n': valid.sum(), 'mean': values[valid].mean(),
                 'bottom_track_mean': through_water['forward'].values[valid].mean(),
                 'bias': difference.mean(), 'rms_difference': np.sqrt((difference ** 2).mean())})
print(f"{len(through_water)} bottom-tracked ensembles in steady flight; "
      f"mean sideways velocity through the water {through_water['sideways'].mean() * 100:.2f} cm/s")
pd.DataFrame(rows).set_index('estimate').round(3)

### Construct the Profiles Both Ways
- **Method 1** references the profiles to the depth-averaged current.
- **Method 2** references them to bottom tracking only (`dac_weight=0`), so they are independent of the depth-averaged current. Only the dives with bottom tracking can be solved.

In both, dives where the DVL cells don't link every depth bin together (`n_segments > 1`, mostly deep dives with few scatterers) aren't solved: the flight model alone links their segments, and its errors add up over depth.

In [ ]:
profiles_dac, ground_dac = inversion.invert_profiles(glider, steady=steady)
profiles_bt, ground_bt = inversion.invert_profiles(
    glider, steady=steady, dac_weight=0, bottom_track_variables=BT_VARIABLES
)

for name, profiles in [('Method 1 (depth-averaged current)', profiles_dac), ('Method 2 (bottom tracking)', profiles_bt)]:
    solved = int(profiles['eastward_seawater_velocity'].notnull().any('depth').sum())
    print(f"{name}: solved {solved} of {profiles.sizes['profile']} dives")

### Check 3: the Frame of the Velocities Against the GPS
The depth-averaged current from the GPS is relative to true north. The time-mean of the Method 2 profiles at the glider (independent of the depth-averaged current) is compared with it for each surfacing, and the rotation between them is fitted. With OOINet's rotated velocities, this rotation is about -14 degrees.

In [ ]:
# The flight model speed over the dives and climbs (glider dataset), as the depth-averaged current is
dac_check = inversion.compare_depth_averaged_current(ground_bt, surfacings, ds=glider_ts, steady=steady_ts)

profile_mean = (dac_check['profile_east'] + 1j * dac_check['profile_north']).to_numpy()
dac = (dac_check['dac_east'] + 1j * dac_check['dac_north']).to_numpy()
usable = (np.abs(profile_mean - dac) < 0.15) & (np.abs(dac) > 0.05)
rotation = np.rad2deg(np.angle(dac[usable] / profile_mean[usable]))
print(f"Rotation from the profiles to the depth-averaged current (counterclockwise positive), "
      f"{usable.sum()} surfacings: median {np.median(rotation):.1f} degrees, "
      f"interquartile range {np.percentile(rotation, 25):.1f} to {np.percentile(rotation, 75):.1f}")

### Check 4: the Depth-Averaged Current Against Method 2
For each surfacing with Method 2 profiles for at least 90% of its DVL measurements, the time-mean of the profiles at the glider is compared with the depth-averaged current. The depth-averaged current is also recalculated with the flight model's speed through the water instead of the glider's own (`m_speed`) over the dives and the climbs (the glider dataset), time-weighted over the steady flight of each segment between surfacings. For comparison, it is also recalculated over the dives only (the DVL times).

In [ ]:
difference = np.hypot(dac_check['difference_east'], dac_check['difference_north'])
flight_difference = np.hypot(dac_check['profile_east'] - dac_check['dac_flight_east'],
                             dac_check['profile_north'] - dac_check['dac_flight_north'])
print(f"{len(dac_check)} surfacings")
print(f"Method 2 minus depth-averaged current: mean east {dac_check['difference_east'].mean():.3f}, "
      f"north {dac_check['difference_north'].mean():.3f} m/s; median magnitude {difference.median():.3f} m/s")
print(f"Method 2 minus depth-averaged current with the flight model speed (dives and climbs): "
      f"median magnitude {flight_difference.median():.3f} m/s")
dives_check = inversion.compare_depth_averaged_current(ground_bt, surfacings, ds=glider, steady=steady)
dives_difference = np.hypot(dives_check['profile_east'] - dives_check['dac_flight_east'],
                            dives_check['profile_north'] - dives_check['dac_flight_north'])
print(f"Method 2 minus depth-averaged current with the flight model speed (dives only): "
      f"median magnitude {dives_difference.median():.3f} m/s")
dac_check.set_index('surface_time').drop(columns='dive_start_time').round(3)

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(9, 4.5))
for ax, component in zip(axes, ['east', 'north']):
    ax.scatter(dac_check[f'dac_{component}'] * 100, dac_check[f'profile_{component}'] * 100,
               s=20, color=BT_COLOR, edgecolor='white', linewidth=0.5)
    limit = 100 * max(np.abs(dac_check[[f'dac_{component}', f'profile_{component}']]).quantile(0.98))
    ax.plot([-limit, limit], [-limit, limit], color='0.3', linewidth=1)
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect('equal')
    ax.set_xlabel(f'Depth-averaged current, {component} (cm s$^{{-1}}$)')
    ax.set_ylabel(f'Method 2 time-mean, {component} (cm s$^{{-1}}$)')
    ax.grid(alpha=0.3)
fig.suptitle(f'{glider_name} deployment {deployment}: each surfacing')
fig.tight_layout()

### Check 5: Method 1 Against Bottom Tracking
The difference of each dive is split into its depth-mean (`offset`), the error of the reference to the depth-averaged current, and the rest (`shape_rms`), the error of the shape of the profile.

In [ ]:
comparison = bottom_track.compare_profiles(profiles_dac, bt_profiles)
offset = np.hypot(comparison['eastward_seawater_velocity_offset'], comparison['northward_seawater_velocity_offset']).to_series().dropna()
print(f"{len(offset)} dives: offset magnitude median {offset.median():.3f}, 90th percentile {offset.quantile(0.9):.3f} m/s")
print(f"Shape rms median: east {float(comparison['eastward_seawater_velocity_shape_rms'].median()):.3f}, "
      f"north {float(comparison['northward_seawater_velocity_shape_rms'].median()):.3f} m/s")

The offsets of each dive are larger than the differences of the surfacings in check 4, because each surfacing averages several dives (about 7, over about 3 hours) and the current changes between them. Method 1 gives every dive between two surfacings the same reference, so it can't follow these changes; the offsets of dives less than an hour apart are similar, and the time-mean of the offsets over each surfacing is about the size of the differences in check 4.

In [ ]:
fig, axes = plt.subplots(nrows=2, figsize=(10, 5.5), sharex=True)
times = profiles_dac.swap_dims(profile='profile_id')['time'].sel(profile_id=comparison['profile_id'])
for ax, component in zip(axes, ['eastward', 'northward']):
    ax.plot(times, comparison[f'{component}_seawater_velocity_offset'] * 100, '.', markersize=4, color=DAC_COLOR)
    ax.axhline(0, color='0.3', linewidth=1)
    ax.set_ylabel(f'{component.capitalize()} offset\n(cm s$^{{-1}}$)')
    ax.set_ylim(-25, 25)
    ax.grid(alpha=0.3)
axes[0].set_title(f'{glider_name} deployment {deployment}: Method 1 minus bottom tracking, each dive')
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(9, 6), sharey=True)
enough = comparison['n_profiles'] >= 10
for ax, component in zip(axes, ['eastward', 'northward']):
    bias = comparison[f'{component}_seawater_velocity_bias'].where(enough) * 100
    rms = comparison[f'{component}_seawater_velocity_rms'].where(enough) * 100
    ax.plot(bias, comparison['depth'], color=DAC_COLOR, linewidth=2, label='Bias')
    ax.plot(rms, comparison['depth'], color=BT_COLOR, linewidth=2, label='RMS')
    ax.axvline(0, color='0.3', linewidth=1)
    ax.set_xlabel(f'Method 1 minus bottom tracking, {component} (cm s$^{{-1}}$)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Depth (m)')
axes[0].set_ylim(float(comparison['depth'].where(enough).max()) + 5, 0)
axes[0].legend()
fig.suptitle(f'{glider_name} deployment {deployment}: depth bins with at least 10 dives')

### Check 6: Carrying the Bottom-Track Reference Up the Water Column
Method 2 takes its reference from the bottom tracking near the end of each dive, and the DVL shear and the flight model carry it up to the rest of the profile. To test how well, Method 2 is solved again with only the deepest 20 m of each dive's bottom tracking (`split_by_depth`), and its glider velocity over the ground is compared with the bottom tracking held out above it, by the height above the reference. The error at the largest heights shows how fast it grows; the upper half of most dives, above any bottom tracking, is only tested on average, by check 4.

In [ ]:
has_bt = np.isfinite(glider['bottom_track_eastward_velocity'].values) & steady
reference, height = bottom_track.split_by_depth(glider, has_bt, span=20)

_, ground_deep = inversion.invert_profiles(
    glider, steady=steady, dac_weight=0, bottom_track_variables=BT_VARIABLES, use_bottom_track=reference
)
held_out = bottom_track.compare_glider_velocity(glider, ground_deep)
index = np.searchsorted(glider['time'].values, held_out['time'].values)
held_out = held_out.assign(height=height[index], profile_id=glider['profile_id'].values[index])
held_out = held_out[(has_bt & ~reference)[index]]
held_out['magnitude'] = np.hypot(held_out['difference_eastward'], held_out['difference_northward'])

edges = np.arange(0, 70, 10)
by_height = held_out.groupby(pd.cut(held_out['height'], edges), observed=True)
carry = pd.DataFrame({
    'dives': by_height['profile_id'].nunique(),
    'ensembles': by_height.size(),
    'median (cm/s)': by_height['magnitude'].median() * 100,
    '90th percentile (cm/s)': by_height['magnitude'].quantile(0.9) * 100,
    'mean east (cm/s)': by_height['difference_eastward'].mean() * 100,
    'mean north (cm/s)': by_height['difference_northward'].mean() * 100,
})
carry.index.name = 'height above the reference (m)'
print(f"{len(held_out)} held-out ensembles in {held_out['profile_id'].nunique()} dives")
carry.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
middle = [interval.mid for interval in carry.index]
ax.plot(carry['median (cm/s)'], middle, '-o', color=BT_COLOR, linewidth=2, markersize=6, label='Median')
ax.plot(carry['90th percentile (cm/s)'], middle, '-o', color=DAC_COLOR, linewidth=2, markersize=6,
        label='90th percentile')
ax.set_xlim(0, None)
ax.set_ylim(0, edges[-1])
ax.set_xlabel('Glider velocity over the ground minus bottom tracking (cm s$^{-1}$)')
ax.set_ylabel('Height above the bottom-track reference (m)')
ax.grid(alpha=0.3)
ax.legend(frameon=False)
ax.set_title(f'{glider_name} deployment {deployment}: Method 2 referenced to the deepest 20 m')

### Example Dive

In [ ]:
dive = 731
methods = [
    (profiles_dac, 'eastward_seawater_velocity', 'Method 1: depth-averaged current', DAC_COLOR),
    (profiles_bt, 'eastward_seawater_velocity', 'Method 2: bottom tracking reference', BT_COLOR),
    (bt_profiles, 'bt_eastward_seawater_velocity_mean', 'Bottom-tracked DVL', OBS_COLOR),
]
fig, axes = plt.subplots(ncols=2, figsize=(9, 6), sharey=True)
deepest = 0
for profiles, east, label, color in methods:
    profile = profiles.swap_dims(profile='profile_id').sel(profile_id=dive)
    for ax, name in zip(axes, [east, east.replace('eastward', 'northward')]):
        ax.plot(profile[name] * 100, profile['depth'], color=color, linewidth=2, label=label)
    deepest = max(deepest, float(profile['depth'].where(profile[east].notnull()).max()))
for ax, component in zip(axes, ['Eastward', 'Northward']):
    ax.set_xlabel(f'{component} velocity (cm s$^{{-1}}$)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Depth (m)')
axes[0].set_ylim(deepest + 5, 0)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False)
fig.subplots_adjust(bottom=0.17)
fig.suptitle(f'{glider_name} deployment {deployment}: dive {dive}')

### Save the Results

In [ ]:
base = f"../data/processed/{glider_name}.deployment{dep_str}"
profiles_dac.to_netcdf(f"{base}.velocity_profiles_dac.nc")
profiles_bt.to_netcdf(f"{base}.velocity_profiles_bottom_track.nc")
bt_profiles.to_netcdf(f"{base}.bottom_track_profiles.nc")
calibration.to_csv(f"{base}.flight_calibration.csv", index=False)
dac_check.to_csv(f"{base}.depth_averaged_current_check.csv", index=False)